In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# # Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# # Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# # kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# import pandas as pd
# data = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
# data.to_csv("submission.csv", index=False)

**Import modules**

In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

In [ ]:
TRAIN_PATH  = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH   = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
OUTPUT_PATH = "/kaggle/working/submission.csv"
 


LABELS      = ["A", "B", "C", "D", "E"]
PRIOR_ORDER = ["B", "C", "A", "D", "E"]   # from training answer distribution
MARGIN_THRESHOLD = 2.0   # min score gap to trust answer retrieval for rank-2


In [ ]:
def full_text(row):
    return str(row["prompt"]) + " " + " ".join(str(row[c]) for c in LABELS)


print("Loading data...")
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)
print(f"Train : {train_df.shape}   Test : {test_df.shape}")

train_rows = [row for _, row in train_df.iterrows()]
test_rows  = [row for _, row in test_df.iterrows()]

In [ ]:
print("\nBuilding answer frequency lookup...")
correct_counts = defaultdict(int)
wrong_counts   = defaultdict(int)

for row in train_rows:
    correct_lbl = row["answer"]
    for lbl in LABELS:
        txt = str(row[lbl]).strip()
        if lbl == correct_lbl:
            correct_counts[txt] += 1
        else:
            wrong_counts[txt] += 1

def answer_retrieval_scores(row):
    return np.array([
        correct_counts.get(str(row[lbl]).strip(), 0)
        - 0.5 * wrong_counts.get(str(row[lbl]).strip(), 0)
        for lbl in LABELS
    ], dtype=float)

In [ ]:
print("Building TF-IDF index...")
train_texts = [full_text(r) for r in train_rows]
test_texts  = [full_text(r) for r in test_rows]

vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True,
                             max_features=100000)
vectorizer.fit(train_texts + test_texts)
train_mat = vectorizer.transform(train_texts)
test_mat  = vectorizer.transform(test_texts)

print("Computing similarity matrix...")
sims_matrix = cosine_similarity(test_mat, train_mat)

**Predictions & submission**

In [ ]:
print("\nGenerating predictions...")
predictions = []

for ti, test_row in enumerate(tqdm(test_rows, desc="Predicting")):
    row_sims = sims_matrix[ti]

    # ── Rank-1: question retrieval
    best_j = int(np.argmax(row_sims))
    rank1  = train_rows[best_j]["answer"]
    r1_idx = LABELS.index(rank1)

    # ── Rank-2 and Rank-3
    # Get answer retrieval scores, zero out rank-1
    a_scores = answer_retrieval_scores(test_row)
    a_scores[r1_idx] = -999999.0

    # Check if there is a CLEAR winner among remaining options
    remaining_vals = a_scores[a_scores > -999999.0]
    sorted_vals    = np.sort(remaining_vals)[::-1]
    margin         = sorted_vals[0] - sorted_vals[1] if len(sorted_vals) >= 2 else 0.0

    if margin > MARGIN_THRESHOLD:
        # Clear winner: trust answer retrieval for rank-2
        r2_idx = int(np.argmax(a_scores))
        rank2  = LABELS[r2_idx]
        # Rank-3: prior order from what remains
        rank3  = next(l for l in PRIOR_ORDER if l != rank1 and l != rank2)
    else:
        # No clear winner: fall back to prior order for rank-2 and rank-3
        rank2 = next(l for l in PRIOR_ORDER if l != rank1)
        rank3 = next(l for l in PRIOR_ORDER if l != rank1 and l != rank2)

    predictions.append({"ID":         test_row["id"],
                         "Prediction": f"{rank1} {rank2} {rank3}"})

submission = pd.DataFrame(predictions)
submission.to_csv(OUTPUT_PATH, index=False)

from collections import Counter
all_p = [p.split() for p in submission["Prediction"]]
print(f"\nRank-1 : {dict(Counter(p[0] for p in all_p))}")
print(f"Rank-2 : {dict(Counter(p[1] for p in all_p))}")
print(f"Rank-3 : {dict(Counter(p[2] for p in all_p))}")
print(f"\nSaved  : {OUTPUT_PATH}")
print(f"Rows   : {len(submission)}")
print(submission.head(10).to_string(index=False))

In [3]:
# train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
# test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")


In [4]:
# print(train_df.shape)
# print(test_df.shape)

(2000, 8)
(500, 7)


In [7]:
# print(train_df.columns.tolist())
# print(test_df.columns.tolist())

['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']
['id', 'prompt', 'A', 'B', 'C', 'D', 'E']
